# 人民银行政策文本：结构化全流程（暂不聚合文章分数）

本 notebook 只读取 `Desktop/project/textmining` 内现有资源，生成 `structured_events` 并覆盖导出结构化事件 CSV。它完成文档日期匹配、硬/软切分、政策—方向—程度—实施状态—否定识别及事件/单句评分，但**暂不把事件聚合为文章总分**。

核心口径：鹰、鸽分别以非负分量保存；`+1/-1` 只用于政策对象标签与方向标签的乘积判别。中性政策同时记 `lambda` 个鹰和 `lambda` 个鸽。否定命中暂令事件权重为 0。

In [2]:
from pathlib import Path
import sys, re, math, warnings
from collections import defaultdict

ROOT = Path('/Users/xiaohan/Desktop/project/textmining')

import numpy as np
import pandas as pd
import jieba

# ===== 可手调参数 =====
NEUTRAL_LAMBDA = 0.10          # 中性政策：同时贡献0.1个鹰与0.1个鸽（仅事件分量，不在本阶段汇总）
INTENSITY_MIN = 0.50           # 最终程度词典权重下界
INTENSITY_MAX = 1.00           # 最终程度词典权重上界
NEGATION_WEIGHT = 0.0          # 当前约定：否定命中则事件计分权重为0
DEFAULT_UNMARKED_INTENSITY = 0.725 # 只有数值程度，或文本/数值程度均缺失时的暂定权重
MAX_POLICY_DIRECTION_GAP = 28  # 同一软分句中政策与方向的最大字符距离
INCLUDE_PENDING_ROWS = True    # 尚未人工标为“保留”时仍加载，输出中明确标记审核状态

POLICY_BOOK = ROOT / '政策——方向.xlsx'
INTENSITY_BOOK = ROOT / '程度词最终词典.xlsx'
STATUS_BOOK = ROOT / '政策实施状态词典_人工审核.xlsx'
NEGATION_FILE = ROOT / '否定词.txt'
REPORT_META = ROOT / '央行沟通交流文本数据（2001-2026）' / '货币政策执行报告.xlsx'
REPORT_TXT = ROOT / '央行沟通交流文本数据（2001-2026）' / '货币政策执行报告TXT'
MEETING_CSV = ROOT / 'monetary_policy_meetings' / 'monetary_policy_meetings.csv'
MEETING_LINKS = ROOT / 'monetary_policy_meetings' / 'meeting_links.csv'
MEETING_FAILED = ROOT / 'monetary_policy_meetings' / 'failed_urls.csv'
MEETING_TXT = ROOT / 'monetary_policy_meetings' / 'raw_txt'
print('项目目录:', ROOT)

项目目录: /Users/xiaohan/Desktop/project/textmining


## 1. 通用读取与审核口径

若工作簿已有人工审核为“保留”的行，优先只用保留项；如果整张表尚无保留项且 `INCLUDE_PENDING_ROWS=True`，则暂用全部候选，并在结构输出中保留原审核状态。

In [3]:
def reviewed_rows(df, review_col='人工审核'):
    if review_col not in df.columns: return df.copy()
    status = df[review_col].fillna('').astype(str).str.strip()
    if (status == '保留').any(): return df[status == '保留'].copy()
    if INCLUDE_PENDING_ROWS:
        warnings.warn(f'{review_col}尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。')
        return df.copy()
    return df.iloc[0:0].copy()

def split_aliases(value):
    if pd.isna(value): return []
    return [x.strip() for x in re.split(r'[；;、,/|]', str(value)) if x.strip()]

def read_text(path):
    for enc in ('utf-8','utf-8-sig','gb18030'):
        try: return path.read_text(encoding=enc)
        except UnicodeDecodeError: pass
    return path.read_text(encoding='utf-8', errors='ignore')

def clean_text(text):
    # 仅移除版面垃圾，不破坏原句顺序；原始文件路径始终保留。
    text = re.sub(r'=+\s*第?\s*\d+\s*页\s*=+', '。', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    return re.sub(r'[\u200b\ufeff\xa0\s]+', ' ', text).strip()

def quarter_end(year, quarter):
    month = quarter * 3
    return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)

## 2. 读取政策对象、方向词、中性项、程度词、实施状态与否定词

程度词直接读取最终词典中统一 RankSVM 生成的 `intensity_weight_0_5_1`，取值范围为 `[0.5, 1]`。文本程度与数值程度均未识别时，结构字段写为 `NA`，不再自动填入1.0。

In [4]:
policy_raw = pd.read_excel(POLICY_BOOK, sheet_name='01政策List')
direction_raw = pd.read_excel(POLICY_BOOK, sheet_name='02方向词')
neutral_raw = pd.read_excel(POLICY_BOOK, sheet_name='05待定条件项')
intensity_raw = pd.read_excel(INTENSITY_BOOK, sheet_name='01最终程度词典')
status_raw = pd.read_excel(STATUS_BOOK, sheet_name='01实施状态词典')

policy_df = reviewed_rows(policy_raw)
direction_df = reviewed_rows(direction_raw)
neutral_df = reviewed_rows(neutral_raw)
status_df = reviewed_rows(status_raw)

policy_terms=[]
for _,r in policy_df.iterrows():
    names=[str(r['政策对象标准名']).strip()]+split_aliases(r.get('别名/匹配词'))
    for term in dict.fromkeys(names):
        policy_terms.append({'term':term,'policy':str(r['政策对象标准名']).strip(),'object_label':int(r['对象标签']),
                             'policy_category':r.get('政策类别'),'object_role':r.get('对象角色'),'policy_review':r.get('人工审核','')})
neutral_terms=[]
for _,r in neutral_df.iterrows():
    names=[str(r['条件项标准名']).strip()]+split_aliases(r.get('别名/匹配词'))
    for term in dict.fromkeys(names):
        neutral_terms.append({'term':term,'policy':str(r['条件项标准名']).strip(),'object_label':0,
                              'policy_category':r.get('类别'),'object_role':r.get('对象角色'),'policy_review':r.get('人工审核','')})
policy_terms = sorted(policy_terms + neutral_terms, key=lambda x:len(x['term']), reverse=True)

direction_terms=[]
for _,r in direction_df.iterrows():
    names=[str(r['方向词标准名']).strip()]+split_aliases(r.get('别名/变体'))
    for term in dict.fromkeys(names): direction_terms.append({'term':term,'direction':str(r['方向词标准名']).strip(),'direction_label':int(r['方向标签']),'direction_review':r.get('人工审核','')})
direction_terms=sorted(direction_terms,key=lambda x:len(x['term']),reverse=True)

# 最终词典已经完成统一RankSVM评分及[0.5,1]归一化，本notebook不再二次缩放。
intensity_terms=[]
for _,r in intensity_raw.iterrows():
    term=str(r['term']).strip(); w=pd.to_numeric(r.get('intensity_weight_0_5_1'),errors='coerce')
    intensity_terms.append({'term':term,'intensity_weight':w,'intensity_source':'final_union_ranksvm_0_5_1' if pd.notna(w) else 'missing_complete_embedding',
                            'rank_score':r.get('rank_score'),'is_training_seed':r.get('is_training_seed'),
                            'max_seed_cosine':r.get('max_seed_cosine'),'score_status':r.get('score_status')})
intensity_terms=sorted(intensity_terms,key=lambda x:len(x['term']),reverse=True)

status_terms=[]
for _,r in status_df.iterrows():
    status_terms.append({'term':str(r['触发短语']).strip(),'implementation_status':r['实施状态'],
                         'implementation_weight':float(r['状态基础权重'])*float(r.get('确定性权重',1)),
                         'implementation_review':r.get('人工审核','')})
status_terms=sorted(status_terms,key=lambda x:len(x['term']),reverse=True)

negation_terms=sorted({x.strip() for x in read_text(NEGATION_FILE).splitlines() if x.strip() and not x.lstrip().startswith('#')},key=len,reverse=True)
for item in policy_terms+direction_terms+intensity_terms+status_terms:
    jieba.add_word(item['term'],freq=2000000)

def build_matcher(items):
    # 联合正则按词长降序，一次扫描完成最长匹配，避免全语料逐词循环。
    item_map={}
    for x in items: item_map.setdefault(x['term'],x)
    terms=sorted(item_map,key=len,reverse=True)
    return re.compile('|'.join(re.escape(x) for x in terms)), item_map
policy_matcher=build_matcher(policy_terms)
direction_matcher=build_matcher(direction_terms)
intensity_matcher=build_matcher(intensity_terms)
status_matcher=build_matcher(status_terms)
negation_matcher=build_matcher([{'term':x} for x in negation_terms])
print({'政策/中性匹配项':len(policy_terms),'方向匹配项':len(direction_terms),'程度候选':len(intensity_terms),'实施状态':len(status_terms),'否定词':len(negation_terms)})

/var/folders/cm/l6n3xg5n2_j3yrpzqmpbm17m0000gn/T/ipykernel_64567/2588083696.py:6: UserWarning: 人工审核尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。
  warnings.warn(f'{review_col}尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。')
/var/folders/cm/l6n3xg5n2_j3yrpzqmpbm17m0000gn/T/ipykernel_64567/2588083696.py:6: UserWarning: 人工审核尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。
  warnings.warn(f'{review_col}尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。')
/var/folders/cm/l6n3xg5n2_j3yrpzqmpbm17m0000gn/T/ipykernel_64567/2588083696.py:6: UserWarning: 人工审核尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。
  warnings.warn(f'{review_col}尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。')
/var/folders/cm/l6n3xg5n2_j3yrpzqmpbm17m0000gn/T/ipykernel_64567/2588083696.py:6: UserWarning: 人工审核尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。
  warnings.warn(f'{review_col}尚无“保留”项：暂加载待审核候选；正式运行前应人工审核。')
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/cm/l6n3xg5n2_j3yrpzqmpbm17m0000gn/T/jieba.cache
Loading model cost 0.425 seconds.
Prefix dict has been built successfully.


{'政策/中性匹配项': 296, '方向匹配项': 60, '程度候选': 146, '实施状态': 60, '否定词': 71}


## 3. 建立文档表并匹配截止日期、发布日期

`cutoff_date` 是报告所对应季度的季末日；`publish_date` 是人民银行实际发布日期。执行报告发布日期来自本地报告元数据工作簿，例会来自下载清单。重复季度文本优先保留正文较长者。

In [5]:
CN_Q={'第一':1,'第二':2,'第三':3,'第四':4}
def parse_period(name, kind):
    if kind=='例会':
        m=re.search(r'(20\d{2})_Q([1-4])',name)
        return (int(m.group(1)),int(m.group(2))) if m else None
    m=re.search(r'(20\d{2})年(第一|第二|第三|第四)季度',name)
    return (int(m.group(1)),CN_Q[m.group(2)]) if m else None

report_meta=pd.read_excel(REPORT_META,sheet_name='货币政策执行报告1')
report_meta['period']=report_meta['标题'].astype(str).map(lambda x:parse_period(x,'执行报告'))
report_date={p:pd.to_datetime(d,errors='coerce') for p,d in zip(report_meta['period'],report_meta['日期']) if p}
# 主文本表缺行时，以链接清单/失败下载清单中的发布日期补齐（例如2024Q4）。
meeting_frames=[pd.read_csv(p,encoding='utf-8-sig') for p in [MEETING_CSV,MEETING_LINKS,MEETING_FAILED] if p.exists()]
meeting_meta=pd.concat(meeting_frames,ignore_index=True).drop_duplicates(['year','quarter'],keep='first')
meeting_date={(int(r.year),int(r.quarter)):pd.to_datetime(r.publish_date,errors='coerce') for r in meeting_meta.itertuples()}

def collect_docs(folder,kind,date_map):
    grouped={}
    for p in folder.glob('*.txt'):
        per=parse_period(p.name,kind)
        if not per: continue
        text=clean_text(read_text(p))
        row={'document_id':f'{kind}-{per[0]}Q{per[1]}','year':per[0],'quarter':per[1],
             'cutoff_date':quarter_end(*per),'publish_date':date_map.get(per,pd.NaT),'text_type':kind,
             'title':p.stem,'source_file':p.name,'source_path':str(p),'text':text}
        if per not in grouped or len(text)>len(grouped[per]['text']): grouped[per]=row
    return list(grouped.values())
docs=collect_docs(REPORT_TXT,'执行报告',report_date)+collect_docs(MEETING_TXT,'例会',meeting_date)
documents=pd.DataFrame([{k:v for k,v in d.items() if k!='text'}|{'text_length':len(d['text'])} for d in docs]).sort_values(['cutoff_date','text_type'])
print('文档数:',len(documents),'发布日期缺失:',int(documents.publish_date.isna().sum()))
documents[['document_id','cutoff_date','publish_date','text_type','source_file']].head()

文档数: 162 发布日期缺失: 0


,document_id,cutoff_date,publish_date,text_type,source_file
24,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,2001年第一季度中国货币政策执行报告.txt
93,执行报告-2001Q2,2001-06-30,2001-07-16,执行报告,2001年第二季度中国货币政策执行报告.txt
49,执行报告-2001Q3,2001-09-30,2001-10-29,执行报告,2001年第三季度中国货币政策执行报告.txt
72,执行报告-2001Q4,2001-12-31,2002-03-06,执行报告,2001年第四季度中国货币政策执行报告.txt
2,执行报告-2002Q1,2002-03-31,2002-05-20,执行报告,2002年第一季度中国货币政策执行报告.txt


## 4. 硬切分、软切分与最长词匹配

句号、分号、问号、感叹号用于硬切分；逗号、冒号、顿号用于软切分。软分句是事件识别单位，但保留所属硬句及前后软分句，后续可增加跨分句继承。政策与方向必须在同一软分句，且字符距离不超过参数阈值。

In [6]:
HARD_SPLIT=re.compile(r'[^。！？!?；;]+[。！？!?；;]?')
SOFT_SPLIT=re.compile(r'[^，,：:、]+[，,：:、]?')
def longest_hits(text,matcher):
    pattern,item_map=matcher
    return [item_map[m.group()]|{'start':m.start(),'end':m.end()} for m in pattern.finditer(text)]

def nearest(hit,candidates,max_gap=None):
    if not candidates:return None
    x=min(candidates,key=lambda y:min(abs(hit['start']-y['end']),abs(y['start']-hit['end'])))
    gap=min(abs(hit['start']-x['end']),abs(x['start']-hit['end']))
    return None if max_gap is not None and gap>max_gap else x

# 数值程度只做结构识别，暂不把不同单位强行换算为统一权重。
NUMERIC_INTENSITY_PATTERN=re.compile(r'(?P<sign>[+-]?)\s*(?P<value>\d+(?:\.\d+)?)\s*(?P<unit>个百分点|个基点|基点|%)')
def numeric_intensity_hits(text):
    return [{'term':m.group(0),'numeric_value':float(m.group('value'))*(-1 if m.group('sign')=='-' else 1),
             'numeric_unit':m.group('unit'),'start':m.start(),'end':m.end()} for m in NUMERIC_INTENSITY_PATTERN.finditer(text)]

clauses=[]
for d in docs:
    for hi,hm in enumerate(HARD_SPLIT.finditer(d['text'])):
        hard=hm.group().strip()
        if not hard:continue
        softs=[m.group().strip(' ，,：:、') for m in SOFT_SPLIT.finditer(hard) if m.group().strip(' ，,：:、')]
        for si,soft in enumerate(softs):
            clauses.append(d|{'hard_sentence_id':f'{d["document_id"]}-H{hi:05d}','soft_clause_id':f'{d["document_id"]}-H{hi:05d}-S{si:02d}',
                              'hard_sentence':hard,'soft_clause':soft,'previous_clause':softs[si-1] if si else '',
                              'next_clause':softs[si+1] if si+1<len(softs) else ''})
print('软分句数:',len(clauses))

软分句数: 247278


## 5. 生成结构化政策事件

每个政策对象生成一条事件记录。`hawk_units` 与 `dove_units` 始终为非负数：方向标签×对象标签只负责决定落在哪一侧；中性项两侧均为 `NEUTRAL_LAMBDA`。程度、实施与否定权重均单列，便于下一步讨论聚合方式。

In [7]:
events=[]
for c in clauses:
    text=c['soft_clause']
    policies=longest_hits(text,policy_matcher)
    if not policies:continue
    directions=longest_hits(text,direction_matcher)
    degrees=longest_hits(text,intensity_matcher)
    numeric_degrees=numeric_intensity_hits(text)
    statuses=longest_hits(text,status_matcher)
    negatives=longest_hits(text,negation_matcher)
    for p in policies:
        direction=nearest(p,directions,MAX_POLICY_DIRECTION_GAP)
        degree=nearest(direction or p,degrees)
        numeric_degree=nearest(direction or p,numeric_degrees)
        status=nearest(direction or p,statuses)
        neg=nearest(direction or p,negatives)
        object_label=int(p['object_label'])
        direction_label=int(direction['direction_label']) if direction else 0
        product=object_label*direction_label
        if object_label==0:
            hawk_units=dove_units=NEUTRAL_LAMBDA; stance='neutral/conditional'
        elif product>0:
            hawk_units,dove_units,stance=1.0,0.0,'hawk'
        elif product<0:
            hawk_units,dove_units,stance=0.0,1.0,'dove'
        else:
            hawk_units=dove_units=0.0;stance='direction_missing'
        degree_weight_value=degree.get('intensity_weight') if degree else np.nan
        if pd.isna(degree_weight_value): degree_weight_value=np.nan
        no_intensity=(degree is None and numeric_degree is None)
        neg_weight=NEGATION_WEIGHT if neg else 1.0
        implementation_weight=status.get('implementation_weight') if status else 1.0
        events.append({
          'document_id':c['document_id'],'cutoff_date':c['cutoff_date'],'publish_date':c['publish_date'],'text_type':c['text_type'],
          'title':c['title'],'source_file':c['source_file'],'source_path':c['source_path'],
          'hard_sentence_id':c['hard_sentence_id'],'soft_clause_id':c['soft_clause_id'],'hard_sentence':c['hard_sentence'],
          'soft_clause':text,'previous_clause':c['previous_clause'],'next_clause':c['next_clause'],
          'policy':p['policy'],'policy_match':p['term'],'policy_start':p['start'],'policy_end':p['end'],
          'policy_category':p.get('policy_category'),'object_role':p.get('object_role'),'object_label':object_label,'policy_review':p.get('policy_review'),
          'direction':direction.get('direction') if direction else None,'direction_match':direction.get('term') if direction else None,
          'direction_start':direction.get('start') if direction else None,'direction_end':direction.get('end') if direction else None,
          'direction_label':direction_label,'direction_review':direction.get('direction_review') if direction else None,'label_product_for_routing':product,
          'stance_route':stance,'hawk_units':hawk_units,'dove_units':dove_units,'neutral_lambda':NEUTRAL_LAMBDA if object_label==0 else 0.0,
          'text_intensity':degree.get('term') if degree else 'NA','text_intensity_weight':degree_weight_value if degree else 'NA',
          'numeric_intensity':numeric_degree.get('term') if numeric_degree else 'NA',
          'numeric_intensity_value':numeric_degree.get('numeric_value') if numeric_degree else 'NA',
          'numeric_intensity_unit':numeric_degree.get('numeric_unit') if numeric_degree else 'NA',
          'intensity':degree.get('term') if degree else (numeric_degree.get('term') if numeric_degree else 'NA'),
          'intensity_weight':degree_weight_value if degree else 'NA',
          'intensity_source':degree.get('intensity_source') if degree else ('numeric_detected_weight_pending' if numeric_degree else 'NA'),
          'intensity_missing':no_intensity,
          'intensity_raw_rank_score':degree.get('rank_score') if degree else 'NA',
          'intensity_is_training_seed':degree.get('is_training_seed') if degree else 'NA',
          'intensity_max_seed_cosine':degree.get('max_seed_cosine') if degree else 'NA',
          'intensity_score_status':degree.get('score_status') if degree else 'NA',
          'implementation_trigger':status.get('term') if status else None,'implementation_status':status.get('implementation_status') if status else 'unknown',
          'implementation_weight':implementation_weight,'implementation_review':status.get('implementation_review') if status else None,
          'negation_present':bool(neg),'negation_term':neg.get('term') if neg else None,'negation_weight':neg_weight,
          # 无可用文本程度权重时预览保持NaN；数值程度的统一换算留待后续讨论。
          'hawk_component_preview':hawk_units*degree_weight_value*implementation_weight*neg_weight if pd.notna(degree_weight_value) else np.nan,
          'dove_component_preview':dove_units*degree_weight_value*implementation_weight*neg_weight if pd.notna(degree_weight_value) else np.nan
        })
structured_events=pd.DataFrame(events).sort_values(['publish_date','document_id','hard_sentence_id','soft_clause_id'],na_position='last').reset_index(drop=True)
print('结构化政策事件数:',len(structured_events),'字段数:',len(structured_events.columns))

结构化政策事件数: 42419 字段数: 54


## 6. 事件/单句评分（暂不聚合文章）

方向缺失的事件不进入评分；否定命中时鹰、鸽贡献均为0；有文本程度时使用最终词典权重，只有数值程度或两类程度均缺失时暂用0.725；实施状态未知时取1.0。评分仍保留在事件/软分句层面，不进行文章聚合。

In [8]:
# 方向缺失只作为结构化匹配记录保留，不进入鹰鸽评分。
structured_events['score_eligible']=(structured_events['direction_label']!=0) & (structured_events['stance_route']!='direction_missing')

def choose_intensity_weight(row):
    if row['text_intensity']!='NA' and pd.notna(pd.to_numeric(row['text_intensity_weight'],errors='coerce')):
        return float(row['text_intensity_weight']), 'text_dictionary_0_5_1'
    if row['numeric_intensity']!='NA':
        return DEFAULT_UNMARKED_INTENSITY, 'numeric_default_0_725'
    return DEFAULT_UNMARKED_INTENSITY, 'missing_both_default_0_725'

chosen=structured_events.apply(choose_intensity_weight,axis=1,result_type='expand')
structured_events['sentence_intensity_weight']=chosen[0]
structured_events['sentence_intensity_source']=chosen[1]
# unknown按当前主口径取1.0；已识别状态沿用实施状态词典权重。
structured_events['sentence_implementation_weight']=np.where(
    structured_events['implementation_status'].eq('unknown'),1.0,
    pd.to_numeric(structured_events['implementation_weight'],errors='coerce').fillna(1.0))
structured_events['sentence_negation_weight']=np.where(structured_events['negation_present'],0.0,1.0)
structured_events['sentence_hawk_score']=np.where(
    structured_events['score_eligible'],
    structured_events['hawk_units']*structured_events['sentence_intensity_weight']*structured_events['sentence_implementation_weight']*structured_events['sentence_negation_weight'],
    np.nan)
structured_events['sentence_dove_score']=np.where(
    structured_events['score_eligible'],
    structured_events['dove_units']*structured_events['sentence_intensity_weight']*structured_events['sentence_implementation_weight']*structured_events['sentence_negation_weight'],
    np.nan)
structured_events['sentence_net_score']=structured_events['sentence_hawk_score']-structured_events['sentence_dove_score']
structured_events['score_exclusion_reason']=np.where(structured_events['score_eligible'],'','direction_missing')

preview_columns=[
 'document_id','cutoff_date','publish_date','text_type','soft_clause',
 'policy','policy_match','object_label','direction','direction_match','direction_label',
 'stance_route','hawk_units','dove_units','neutral_lambda',
 'text_intensity','text_intensity_weight','numeric_intensity','numeric_intensity_value','numeric_intensity_unit',
 'intensity','intensity_weight','intensity_source','intensity_missing',
 'implementation_status','implementation_trigger','implementation_weight',
 'negation_present','negation_term','negation_weight',
 'score_eligible','sentence_intensity_weight','sentence_intensity_source','sentence_implementation_weight',
 'sentence_negation_weight','sentence_hawk_score','sentence_dove_score','sentence_net_score','score_exclusion_reason',
 'hawk_component_preview','dove_component_preview'
]
structured_events[preview_columns].head(10)

,document_id,cutoff_date,publish_date,text_type,soft_clause,policy,policy_match,object_label,direction,direction_match,...,intensity_source,intensity_missing,implementation_status,implementation_trigger,implementation_weight,negation_present,negation_term,negation_weight,hawk_component_preview,dove_component_preview
0,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,一季度货币信贷运行的基本情况 二,货币信贷,货币信贷,-1,NaN,NaN,...,NA,True,unknown,NaN,1.0,False,NaN,1.0,NaN,NaN
1,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,当前货币供应量总体适度,货币供应量,货币供应量,-1,NaN,NaN,...,final_union_ranksvm_0_5_1,False,unknown,NaN,1.0,False,NaN,1.0,0.00000,0.000000
2,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,一季度货币信贷运行的基本情况 今年一季度我国货币信贷继续保持健康平稳的发展态势,货币信贷,货币信贷,-1,NaN,NaN,...,final_union_ranksvm_0_5_1,False,正在实施,继续保持,1.0,False,NaN,1.0,0.00000,0.000000
3,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,一季度货币信贷运行的基本情况 今年一季度我国货币信贷继续保持健康平稳的发展态势,货币信贷,货币信贷,-1,NaN,NaN,...,final_union_ranksvm_0_5_1,False,正在实施,继续保持,1.0,False,NaN,1.0,0.00000,0.000000
4,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,货币供应量适度 增长,货币供应量,货币供应量,-1,增长,增长,...,final_union_ranksvm_0_5_1,False,unknown,NaN,1.0,False,NaN,1.0,0.00000,0.660719
5,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,人民币汇率保持稳定。,人民币汇率水平,人民币汇率,0,NaN,NaN,...,final_union_ranksvm_0_5_1,False,unknown,NaN,1.0,False,NaN,1.0,0.07468,0.074680
6,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,（一）货币供应量平稳增长,货币供应量,货币供应量,-1,增长,增长,...,final_union_ranksvm_0_5_1,False,unknown,NaN,1.0,False,NaN,1.0,0.00000,0.706017
7,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,广义货币M2余额13.9万 亿元,货币供应量,广义货币M2,-1,NaN,NaN,...,NA,True,unknown,NaN,1.0,False,NaN,1.0,NaN,NaN
8,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,货币流动性 M1/M2为38.2%,流动性供给,流动性,-1,NaN,NaN,...,numeric_detected_weight_pending,False,unknown,NaN,1.0,False,NaN,1.0,NaN,NaN
9,执行报告-2001Q1,2001-03-31,2001-05-14,执行报告,货币流动性 M1/M2为38.2%,货币供应量,M2,-1,NaN,NaN,...,numeric_detected_weight_pending,False,unknown,NaN,1.0,False,NaN,1.0,NaN,NaN


### 常用核查指令（按需取消注释）

```python
# 查看某份文档
# structured_events.query("document_id == '例会-2025Q4'")[preview_columns].head(30)

# 查看方向缺失，便于补词典或调整跨软句继承
# structured_events.query("stance_route == 'direction_missing'")[preview_columns].head(30)

# 查看中性政策lambda分量
# structured_events.query("stance_route == 'neutral/conditional'")[preview_columns].head(30)

# 查看否定命中
# structured_events.query('negation_present')[preview_columns].head(30)
```

注意：`hawk_component_preview` 和 `dove_component_preview` 只是事件级乘法检查列，不代表最终文章分数。文章级分母、重复政策去重、当前政策状态与沟通增量的聚合方式留到下一步确定。

In [9]:
from pathlib import Path

OUTPUT_DIR = Path("/Users/xiaohan/Desktop/project/textmining")
OUTPUT_CSV = OUTPUT_DIR / "结构化政策事件_最终输出.csv"

# 创建目录（目录已经存在时不会产生影响）
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 导出最终的政策事件明细
structured_events.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

print("导出完成：", OUTPUT_CSV)
print("数据行数：", len(structured_events))
print("字段数量：", len(structured_events.columns))

导出完成： /Users/xiaohan/Desktop/project/textmining/结构化政策事件_最终输出.csv
数据行数： 42419
字段数量： 54
